# Filtrado de Imágenes — Pipeline Completo

Este notebook recrea el paquete `filtrado/` mediante `%%writefile` y ejecuta
la aplicación Streamlit.  Es un respaldo para ejecución en Google Colab.

**Requisitos**: matplotlib, numpy, opencv-python, streamlit

In [ ]:
# --- Instalación de dependencias ---
!pip install -q opencv-python numpy matplotlib streamlit pyngrok

## 1. Crear el paquete `filtrado/`

Cada celda siguiente escribe un archivo del paquete usando `%%writefile`.
Ejecútelas en orden.

In [ ]:
%%writefile filtrado/__init__.py
"""Image filtering pipeline — core filtering and display utilities."""

from filtrado.core import (
    validate_image,
    crop_center,
    crop_manual,
    mean_filter,
    median_filter,
    laplacian_filter,
    sobel_filter,
)
from filtrado.display import (
    show_histogram,
    show_digitalization_grid,
    show_filter_comparison,
)

__all__ = [
    "validate_image",
    "crop_center",
    "crop_manual",
    "mean_filter",
    "median_filter",
    "laplacian_filter",
    "sobel_filter",
    "show_histogram",
    "show_digitalization_grid",
    "show_filter_comparison",
]

In [ ]:
%%writefile filtrado/core.py
"""Core image filtering operations — pure functions that receive/return np.ndarray."""

import cv2
import numpy as np


def validate_image(img: np.ndarray) -> tuple[bool, str]:
    """Validate image is B&W (single-channel or equal RGB) and ≥ 15×15.

    Returns
    -------
    (is_valid, error_msg)
        is_valid is True when the image passes all checks.
        error_msg describes the rejection reason when is_valid is False.
    """
    # --- size check ---
    if img.shape[0] < 15 or img.shape[1] < 15:
        return (
            False,
            f"Image too small: {img.shape[1]}×{img.shape[0]}. "
            f"Minimum size is 15×15 pixels.",
        )

    # --- B&W check ---
    if img.ndim == 2:
        # Single-channel grayscale — always B&W
        return True, ""

    if img.ndim == 3 and img.shape[2] >= 3:
        # Multi-channel — all channels must be identical
        if (
            np.array_equal(img[:, :, 0], img[:, :, 1])
            and np.array_equal(img[:, :, 0], img[:, :, 2])
        ):
            return True, ""

        return (
            False,
            "Color image detected. Only black-and-white (B&W) images are accepted. "
            "Please upload a grayscale JPG with all channels equal.",
        )

    return False, f"Unsupported image format with ndim={img.ndim}."


def crop_center(img: np.ndarray, size: int = 15) -> np.ndarray:
    """Extract a centered *size* × *size* crop from *img*.

    Handles odd and even dimensions gracefully.  The centre coordinate is
    computed as ``(dim - 1) // 2`` so that the spec's 100 × 100 example
    produces a crop starting at pixel (42, 42).
    """
    h, w = img.shape[:2]
    half = size // 2
    cy, cx = (h - 1) // 2, (w - 1) // 2

    y_start = cy - half
    y_end = y_start + size
    x_start = cx - half
    x_end = x_start + size

    return img[y_start:y_end, x_start:x_end]


def crop_manual(
    img: np.ndarray, x: int, y: int, size: int = 15
) -> np.ndarray:
    """Extract a *size* × *size* crop from *img* starting at pixel (x, y).

    Raises
    ------
    ValueError
        If the requested region extends beyond the image boundaries.
    """
    h, w = img.shape[:2]
    if x + size > w or y + size > h:
        raise ValueError(
            f"Crop region ({size}×{size}) at ({x},{y}) exceeds image "
            f"bounds ({w}×{h})."
        )
    return img[y : y + size, x : x + size]


def mean_filter(img: np.ndarray) -> np.ndarray:
    """Apply a 3×3 mean (averaging) filter.

    Returns an array of the same shape and dtype as the input.
    """
    return cv2.blur(img, (3, 3))


def median_filter(img: np.ndarray) -> np.ndarray:
    """Apply a 3×3 median filter.

    Returns an array of the same shape and dtype as the input.
    """
    return cv2.medianBlur(img, 3)


def laplacian_filter(img: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Apply Laplacian edge detection.

    Returns
    -------
    (raw, normalized)
        raw — cv2.Laplacian result as float64 (can contain negative values).
        normalized — rescaled to [0, 255] range as uint8.
    """
    raw = cv2.Laplacian(img, cv2.CV_64F)
    normalized = cv2.normalize(raw, None, 0, 255, cv2.NORM_MINMAX)
    normalized = normalized.astype(np.uint8)
    return raw, normalized


def sobel_filter(img: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Apply Sobel edge detection (combined XY magnitude).

    Returns
    -------
    (raw, normalized)
        raw — combined gradient magnitude as float64.
        normalized — rescaled to [0, 255] range as uint8.
    """
    grad_x = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)
    grad_y = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)
    raw = cv2.magnitude(grad_x, grad_y)
    normalized = cv2.normalize(raw, None, 0, 255, cv2.NORM_MINMAX)
    normalized = normalized.astype(np.uint8)
    return raw, normalized


In [ ]:
%%writefile filtrado/display.py
"""Display utilities — matplotlib figures for image visualisation."""

import matplotlib.pyplot as plt
import numpy as np


def show_histogram(img: np.ndarray, title: str = "") -> plt.Figure:
    """Return a matplotlib Figure with the pixel intensity distribution.

    Parameters
    ----------
    img : np.ndarray
        Grayscale image (2D array).
    title : str
        Optional title for the plot.
    """
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(img.ravel(), bins=256, range=(0, 256), color="gray", alpha=0.7)
    ax.set_xlabel("Intensidad de pixel")
    ax.set_ylabel("Frecuencia")
    ax.set_xlim(0, 255)
    if title:
        ax.set_title(title)
    fig.tight_layout()
    return fig


def show_digitalization_grid(img: np.ndarray) -> plt.Figure:
    """Return a matplotlib Figure with a pixel grid + numeric cell values.

    Each cell displays its pixel value.  Text colour is chosen automatically
    to contrast with the cell background (white on dark, black on light).
    """
    fig, ax = plt.subplots(figsize=(8, 8))
    h, w = img.shape[:2]

    # Display the image
    ax.imshow(img, cmap="gray", interpolation="nearest")

    # Grid lines at pixel boundaries
    ax.set_xticks(np.arange(-0.5, w, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, h, 1), minor=True)
    ax.grid(True, which="minor", color="gray", linewidth=0.5)
    ax.tick_params(which="minor", length=0)

    # Overlay numeric values
    edge = isinstance(img.ravel()[0], (np.floating, float))
    for i in range(h):
        for j in range(w):
            val = img[i, j]
            text = f"{val:.1f}" if edge else str(val)
            colour = "white" if val < 128 else "black"
            ax.text(j, i, text, ha="center", va="center",
                    color=colour, fontsize=7)

    ax.set_xticks([])
    ax.set_yticks([])
    fig.tight_layout()
    return fig


def show_filter_comparison(
    original: np.ndarray,
    filtered: np.ndarray,
    title: str = "",
) -> plt.Figure:
    """Return a matplotlib Figure comparing *original* vs *filtered*.

    The figure contains a 2×2 grid:
      - top-left:   original image
      - top-right:  filtered image
      - bottom-left:  original histogram
      - bottom-right: filtered histogram
    """
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))

    # --- images ---
    axes[0, 0].imshow(original, cmap="gray", interpolation="nearest")
    axes[0, 0].set_title("Original")
    axes[0, 0].axis("off")

    axes[0, 1].imshow(filtered, cmap="gray", interpolation="nearest")
    axes[0, 1].set_title("Filtrada")
    axes[0, 1].axis("off")

    # --- histograms ---
    axes[1, 0].hist(original.ravel(), bins=256, range=(0, 256),
                    color="gray", alpha=0.7)
    axes[1, 0].set_title("Histograma — original")
    axes[1, 0].set_xlim(0, 255)

    axes[1, 1].hist(filtered.ravel(), bins=256, range=(0, 256),
                    color="gray", alpha=0.7)
    axes[1, 1].set_title("Histograma — filtrada")
    axes[1, 1].set_xlim(0, 255)

    if title:
        fig.suptitle(title)

    fig.tight_layout()
    return fig


In [ ]:
%%writefile app.py
"""Streamlit web app — Image Filtering Pipeline."""

import streamlit as st
import cv2
import numpy as np

from filtrado.core import (
    validate_image,
    crop_center,
    crop_manual,
    mean_filter,
    median_filter,
    laplacian_filter,
    sobel_filter,
)
from filtrado.display import (
    show_digitalization_grid,
    show_filter_comparison,
)

st.set_page_config(layout="wide", page_title="Filtrado de Imágenes")

st.title("Filtrado de Imágenes")
st.markdown(
    "Cargue una imagen **JPG en blanco y negro** para aplicar filtros "
    "de procesamiento digital."
)

uploaded = st.file_uploader(
    "Seleccione una imagen...",
    type=["jpg", "jpeg"],
)

if uploaded is None:
    st.info("Sube una imagen JPG para comenzar.")
    st.stop()

bytes_data = uploaded.getvalue()
img_array = np.frombuffer(bytes_data, np.uint8)
img_color = cv2.imdecode(img_array, cv2.IMREAD_COLOR)

if img_color is None:
    st.error("No se pudo decodificar la imagen. El archivo podría estar corrupto.")
    st.stop()

is_valid, msg = validate_image(img_color)
if not is_valid:
    st.error(f"Imagen inválida: {msg}")
    st.stop()

st.success("Imagen válida — blanco y negro aceptado.")

gray = cv2.cvtColor(img_color, cv2.COLOR_BGR2GRAY)

st.markdown("### Recorte")
crop_option = st.radio(
    "Seleccione modo de recorte:",
    ["Imagen completa", "Recorte 15×15 (centro)", "Recorte 15×15 (manual)"],
    horizontal=True,
)

if crop_option == "Imagen completa":
    cropped = gray
elif crop_option == "Recorte 15×15 (centro)":
    cropped = crop_center(gray)
else:
    col1, col2 = st.columns(2)
    max_x = max(0, gray.shape[1] - 15)
    max_y = max(0, gray.shape[0] - 15)
    with col1:
        x = st.number_input("X (columna)", min_value=0, max_value=max_x, value=0)
    with col2:
        y = st.number_input("Y (fila)", min_value=0, max_value=max_y, value=0)
    try:
        cropped = crop_manual(gray, int(x), int(y))
    except ValueError as e:
        st.error(str(e))
        st.stop()

st.caption(f"DimensiÃ³n final: {cropped.shape[1]}Ã{cropped.shape[0]} pÃxeles")

st.markdown("### Filtro")
filter_option = st.selectbox(
    "Seleccione un filtro:",
    ["Media", "Mediana", "Laplaciano", "Sobel"],
)

is_edge = filter_option in ("Laplaciano", "Sobel")

if st.button("Aplicar filtro", type="primary"):
    st.markdown("---")
    st.subheader("Imagen original")
    orig_col1, orig_col2 = st.columns(2)
    with orig_col1:
        st.image(cropped, caption="Original", clamp=True, use_container_width=True)
    with orig_col2:
        st.pyplot(show_digitalization_grid(cropped))
        st.caption("DigitalizaciÃ³n — matriz de pÃxeles")

    if filter_option == "Media":
        result = mean_filter(cropped)
        raw = None
    elif filter_option == "Mediana":
        result = median_filter(cropped)
        raw = None
    elif filter_option == "Laplaciano":
        raw, result = laplacian_filter(cropped)
    else:
        raw, result = sobel_filter(cropped)

    if is_edge:
        st.subheader("Matrices del filtrado")
        mat_col1, mat_col2 = st.columns(2)
        with mat_col1:
            st.pyplot(show_digitalization_grid(raw))
            st.caption("Matriz resultante del filtrado (raw â€” float64)")
        with mat_col2:
            st.pyplot(show_digitalization_grid(result))
            st.caption("Matriz re-escalada (0â€“255 â€” uint8)")

        st.subheader("Imagen resultante")
        res_col1, res_col2 = st.columns(2)
        with res_col1:
            st.image(result, caption=f"Filtro {filter_option}",
                     clamp=True, use_container_width=True)
        with res_col2:
            st.pyplot(show_digitalization_grid(result))
            st.caption("DigitalizaciÃ³n de la imagen resultante")
    else:
        st.subheader("Imagen resultante")
        res_col1, res_col2 = st.columns(2)
        with res_col1:
            st.image(result, caption=f"Filtro {filter_option}",
                     clamp=True, use_container_width=True)
        with res_col2:
            st.pyplot(show_digitalization_grid(result))
            st.caption("DigitalizaciÃ³n — matriz obtenida tras el filtro")

    st.subheader("ComparaciÃ³n")
    st.pyplot(show_filter_comparison(
        cropped, result, title=f"Original vs {filter_option}"
    ))


## 2. Ejecutar el pipeline

### Opción A: Streamlit + ngrok (recomendado para presentación)

La siguiente celda inicia Streamlit en segundo plano y expone un túnel
ngrok para acceder desde cualquier dispositivo.

In [ ]:
# Iniciar Streamlit en segundo plano y exponer con ngrok
import threading, time
from pyngrok import ngrok

# Lanzar Streamlit en un hilo separado
def run_streamlit():
    !streamlit run app.py --server.port 8501 &>/dev/null &

thread = threading.Thread(target=run_streamlit, daemon=True)
thread.start()
time.sleep(5)  # Esperar a que Streamlit inicie

# Crear túnel público
public_url = ngrok.connect(8501)
print(f"Aplicación disponible en: {public_url}")
print("\nNOTA: La URL de ngrok cambia cada vez que se reinicia Colab."
      " Si necesitas una URL fija, considera usar stlite en su lugar.")

### Opción B: Probar el pipeline en local

Si estás ejecutando este notebook localmente, puedes simplemente abrir
una terminal y ejecutar:

```bash
streamlit run app.py
```

## 3. Verificar la instalación

Si todo está correcto, la siguiente celda ejecuta pruebas rápidas
de las funciones principales.

In [ ]:
# Smoke test — verificar que las funciones importan y devuelven tipos correctos
import numpy as np
from filtrado.core import validate_image, crop_center, mean_filter

# Crear imagen de prueba
test_img = np.random.randint(0, 256, (50, 50), dtype=np.uint8)

# Validar
ok, msg = validate_image(test_img)
assert ok, f"Validate failed: {msg}"
print("✓ validate_image: OK")

# Recortar
cropped = crop_center(test_img, size=15)
assert cropped.shape == (15, 15), f"Expected 15x15, got {cropped.shape}"
print("✓ crop_center: OK")

# Filtrar
result = mean_filter(cropped)
assert result.shape == (15, 15)
print("✓ mean_filter: OK")

print("\nTodos los checks pasaron.")